# PoliMillionaire Game Tests

Run selected local models one after another.

## Setup

In [1]:
import gc
import importlib.util
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "polimillionaire" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the project folder.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
for path in [REPO_ROOT / "src", REPO_ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

os.environ.setdefault("HF_HOME", str(REPO_ROOT / "data" / "hf_home"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(REPO_ROOT / "data" / "hf_home" / "hub"))
print("Repo:", REPO_ROOT)
print("HF cache:", os.environ["HF_HOME"])

Repo: C:\Users\sjuan\Desktop\NLP\project\nlp-polimillionaire
HF cache: C:\Users\sjuan\Desktop\NLP\project\nlp-polimillionaire\data\hf_home


## Settings

Set `run=True` for the models you want to test.

In [2]:
RUN_API_CHECK = False
RUN_LIVE_GAME = True
RUN_OFFLINE_BENCHMARK = True
PROMPT_FOR_CREDENTIALS = False

API_URL = "http://131.175.15.22:51111/"
COMPETITION_ID = 0
SAFE_DELAY_SECONDS = 2.0
ANSWER_TIMEOUT_SECONDS = 25.0

MODELS_TO_RUN = [
    {"label": "Gemma 4 E2B", "kind": "gemma", "model_id": "google/gemma-4-E2B-it", "run": False},
    {"label": "Gemma 4 E4B", "kind": "gemma", "model_id": "google/gemma-4-E4B-it", "run": False},
    {"label": "Qwen3.5 2B Thinking", "kind": "qwen_thinking", "model_id": "Qwen/Qwen3.5-2B", "run": False},
    {"label": "Gemma 4 E2B Council", "kind": "gemma_council", "model_id": "google/gemma-4-E2B-it", "votes": 3, "run": False},
    {"label": "Gemma + Qwen Mixed Council", "kind": "mixed_council", "model_id": "google/gemma-4-E2B-it + Qwen/Qwen3.5-2B", "run": False},
    {"label": "Gemma 4 E2B (4-bit)", "kind": "gemma_quantized", "model_id": "google/gemma-4-E2B-it", "run": True},
    {"label": "Gemma + Qwen Mixed Council (4-bit)", "kind": "mixed_quantized", "model_id": "google/gemma-4-E2B-it + Qwen/Qwen3.5-2B", "run": True},
]

print("API check:", RUN_API_CHECK, "Live game:", RUN_LIVE_GAME, "Benchmark:", RUN_OFFLINE_BENCHMARK)
for item in MODELS_TO_RUN:
    print("-", item["label"], "run=", item["run"])

API check: False Live game: True Benchmark: True
- Gemma 4 E2B run= False
- Gemma 4 E4B run= False
- Qwen3.5 2B Thinking run= False
- Gemma 4 E2B Council run= False
- Gemma + Qwen Mixed Council run= False
- Gemma 4 E2B (4-bit) run= True
- Gemma + Qwen Mixed Council (4-bit) run= True


## Login

In [3]:
import getpass
from millionaire_client import AuthenticationError, MillionaireClient

USERNAME = os.environ.get("POLIMILLIONAIRE_USERNAME")
PASSWORD = os.environ.get("POLIMILLIONAIRE_PASSWORD")
try:
    from google.colab import userdata
    USERNAME = USERNAME or userdata.get("POLIMILLIONAIRE_USERNAME")
    PASSWORD = PASSWORD or userdata.get("POLIMILLIONAIRE_PASSWORD")
except Exception:
    pass
if PROMPT_FOR_CREDENTIALS and not USERNAME:
    USERNAME = input("PoliMillionaire username: ").strip()
if PROMPT_FOR_CREDENTIALS and not PASSWORD:
    PASSWORD = getpass.getpass("PoliMillionaire password: ")
print("Username configured:", bool(USERNAME))
print("Password configured:", bool(PASSWORD))

Username configured: True
Password configured: True


## Competitions

In [4]:
client = None
if RUN_API_CHECK:
    if not USERNAME or not PASSWORD:
        raise RuntimeError("Set credentials first.")
    try:
        client = MillionaireClient(API_URL)
        user = client.login(USERNAME, PASSWORD)
        print("Logged in as", user.username)
        for competition in client.competitions.list_all():
            print(competition.id, competition.name, competition.max_levels)
    except AuthenticationError as exc:
        client = None
        print("Login failed:", exc)
else:
    print("API check skipped.")

API check skipped.


## Run Selected Models

The 4-bit option follows the class quantization recipe and needs `bitsandbytes`.

In [5]:
from polimillionaire.runner import GameRunner, RunLogger
from polimillionaire.strategies import CouncilStrategy, GemmaLLM, GemmaLLMConfig, GemmaStrategy, QwenLLM, QwenLLMConfig, QwenStrategy
from polimillionaire.types import AnswerOption, Question

warmup_question = Question(1, "What is 2 + 2?", [AnswerOption(0, "3"), AnswerOption(1, "4"), AnswerOption(2, "5"), AnswerOption(3, "22")])
BENCHMARK_SET = [
    (warmup_question, 1),
    (Question(2, "Which song was NOT written by Bob Dylan?", [AnswerOption(0, "Like a Rolling Stone"), AnswerOption(1, "Blowin' in the Wind"), AnswerOption(2, "The Times They Are A-Changin'"), AnswerOption(3, "Hound Dog")]), 3),
    (Question(3, "What was Whitney Houston's debut album?", [AnswerOption(0, "Whitney Houston"), AnswerOption(1, "Just Whitney"), AnswerOption(2, "I'm Your Baby Tonight"), AnswerOption(3, "Whitney")]), 0),
]


def clear_model_memory():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception as exc:
        print("Cleanup warning:", exc)


def mixed_council(quantized=False):
    if quantized and importlib.util.find_spec("bitsandbytes") is None:
        raise RuntimeError("For the 4-bit council, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
    gemma = GemmaLLM(GemmaLLMConfig(model_id="google/gemma-4-E2B-it", quantize_4bit=quantized, max_new_tokens=32, do_sample=True, seed=42, generation_max_time_seconds=6.0, timeout_seconds=120.0))
    qwen = QwenLLM(QwenLLMConfig(model_id="Qwen/Qwen3.5-2B", quantize_4bit=quantized, max_new_tokens=32, temperature=0.7, top_p=0.8, top_k=20, do_sample=True, seed=43, enable_thinking=False, generation_max_time_seconds=6.0, timeout_seconds=120.0))
    return CouncilStrategy(candidate_llms=[gemma, qwen], judge_llm=gemma, judge_scope="candidate_only" if quantized else "any_option", rejected_judge_fallback="primary_candidate" if quantized else "confidence_weighted", base_seed=200, candidate_max_new_tokens=32, judge_max_new_tokens=8, max_time_per_call=6.0)


def make_strategy(item):
    if item["kind"] == "gemma_quantized":
        if importlib.util.find_spec("bitsandbytes") is None:
            raise RuntimeError("For 4-bit Gemma, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
        return GemmaStrategy(model_config=GemmaLLMConfig(model_id=item["model_id"], quantize_4bit=True, max_new_tokens=8, temperature=0.0, do_sample=False, num_beams=1, seed=42, generation_max_time_seconds=18.0, timeout_seconds=120.0))
    if item["kind"] == "mixed_quantized":
        return mixed_council(quantized=True)
    if item["kind"] == "mixed_council":
        return mixed_council()
    if item["kind"] == "gemma_council":
        llm = GemmaLLM(GemmaLLMConfig(model_id=item["model_id"], max_new_tokens=48, do_sample=True, seed=42, generation_max_time_seconds=4.5, timeout_seconds=120.0))
        return CouncilStrategy(llm=llm, num_votes=item.get("votes", 3), base_seed=100, candidate_max_new_tokens=48, judge_max_new_tokens=8, max_time_per_call=4.5)
    if item["kind"] == "qwen_thinking":
        return QwenStrategy(model_config=QwenLLMConfig(model_id=item["model_id"], max_new_tokens=128, temperature=1.0, top_p=0.95, top_k=20, do_sample=True, seed=42, enable_thinking=True, generation_max_time_seconds=18.0, timeout_seconds=120.0))
    return GemmaStrategy(model_config=GemmaLLMConfig(model_id=item["model_id"], max_new_tokens=8, temperature=0.0, do_sample=False, num_beams=1, seed=42, generation_max_time_seconds=18.0, timeout_seconds=120.0))


def clean_name(label):
    return "_".join(label.lower().replace("+", " ").replace("(", " ").replace(")", " ").split())


benchmark_results = []


def benchmark(strategy, label):
    rows = []
    for question, gold_id in BENCHMARK_SET:
        started = time.monotonic()
        prediction = strategy.answer(question)
        seconds = time.monotonic() - started
        correct = prediction.option_id == gold_id
        rows.append((correct, seconds, bool(prediction.metadata.get("fallback"))))
        gold = question.require_option(gold_id)
        print("\nQ:", question.text)
        print("predicted:", prediction.option_id, prediction.answer_text)
        print("gold:", gold.id, gold.text, "correct:", correct, "seconds:", round(seconds, 2))
        print("decision:", prediction.metadata.get("decision_method"), "fallback:", prediction.metadata.get("fallback"))
        for index, vote in enumerate(prediction.metadata.get("votes") or [], start=1):
            print("vote", index, vote.get("model_name"), "->", vote.get("option_id"), "confidence:", vote.get("confidence"), "reason:", vote.get("reasoning"))
        if prediction.metadata.get("judge_raw_text") is not None:
            print("judge raw:", prediction.metadata.get("judge_raw_text"), "scope:", prediction.metadata.get("judge_scope"))
    accuracy = sum(correct for correct, _, _ in rows) / len(rows)
    summary = {"model": label, "accuracy": accuracy, "avg_seconds": round(sum(seconds for _, seconds, _ in rows) / len(rows), 2), "fallbacks": sum(fallback for _, _, fallback in rows)}
    benchmark_results.append(summary)
    print("Benchmark summary:", summary)
    return accuracy >= 0.60 and not any(fallback for _, _, fallback in rows)


if RUN_LIVE_GAME and (not USERNAME or not PASSWORD):
    raise RuntimeError("Set credentials first.")
if RUN_LIVE_GAME and client is None:
    client = MillionaireClient(API_URL)
    client.login(USERNAME, PASSWORD)

run_results = []
for item in MODELS_TO_RUN:
    if not item.get("run", False):
        print("Skipped", item["label"])
        continue
    strategy = None
    try:
        print("\nModel:", item["label"])
        strategy = make_strategy(item)
        started = time.monotonic()
        warmup = strategy.answer(warmup_question)
        load_and_warmup_seconds = time.monotonic() - started
        print("warmup option:", warmup.option_id, warmup.answer_text, "fallback:", warmup.metadata.get("fallback"), "load + answer seconds:", round(load_and_warmup_seconds, 2))
        if warmup.metadata.get("fallback") or warmup.option_id != 1:
            raise RuntimeError("Warm-up failed. No live game started.")
        if item["kind"] in {"gemma_quantized", "mixed_quantized"}:
            devices = [model.device_summary for model in strategy.candidate_llms] if isinstance(strategy, CouncilStrategy) else [strategy.llm.device_summary]
            print("devices:", devices)
            if any(token in " ".join(devices).lower() for token in ("'cpu'", "'disk'", "meta")):
                raise RuntimeError("Quantized model was offloaded. No live game started.")
            started = time.monotonic()
            speed_check = strategy.answer(warmup_question)
            answer_seconds = time.monotonic() - started
            print("loaded-model speed check:", round(answer_seconds, 2), "seconds")
            if speed_check.metadata.get("fallback") or speed_check.option_id != 1:
                raise RuntimeError("Quantized model speed check failed. No live game started.")
            if answer_seconds > 20.0:
                raise RuntimeError("Quantized model loaded answer is over 20 seconds. No live game started.")
        benchmark_ok = benchmark(strategy, item["label"]) if RUN_OFFLINE_BENCHMARK else True
        if item["kind"] in {"gemma_quantized", "mixed_quantized"} and RUN_LIVE_GAME and not RUN_OFFLINE_BENCHMARK:
            raise RuntimeError("Set RUN_OFFLINE_BENCHMARK=True before a live 4-bit run.")
        if not benchmark_ok and RUN_LIVE_GAME:
            raise RuntimeError("Benchmark guard failed. No live game started.")
        if not benchmark_ok:
            print("Benchmark below live threshold; kept for offline comparison.")
        result = {"label": item["label"], "warmup_option": warmup.option_id, "benchmark_ok": benchmark_ok}
        if RUN_LIVE_GAME:
            run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
            log_path = REPO_ROOT / "results" / "runs" / f"{clean_name(item['label'])}_{run_id}.jsonl"
            game = GameRunner(client, SAFE_DELAY_SECONDS, ANSWER_TIMEOUT_SECONDS, RunLogger(log_path)).play(COMPETITION_ID, strategy)
            result.update({"earned_amount": game.earned_amount, "log_path": str(log_path)})
            print("Earned amount:", game.earned_amount, "Log path:", log_path)
        run_results.append(result)
    finally:
        del strategy
        clear_model_memory()
print("\nComparison:")
for summary in benchmark_results:
    print(summary)
run_results

Skipped Gemma 4 E2B
Skipped Gemma 4 E4B
Skipped Qwen3.5 2B Thinking
Skipped Gemma 4 E2B Council
Skipped Gemma + Qwen Mixed Council

Model: Gemma 4 E2B (4-bit)


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

warmup option: 1 4 fallback: False load + answer seconds: 22.94
devices: ['cuda:0']
loaded-model speed check: 0.2 seconds

Q: What is 2 + 2?
predicted: 1 4
gold: 1 4 correct: True seconds: 0.2
decision: None fallback: False

Q: Which song was NOT written by Bob Dylan?
predicted: 3 Hound Dog
gold: 3 Hound Dog correct: True seconds: 0.2
decision: None fallback: False

Q: What was Whitney Houston's debut album?
predicted: 3 Whitney
gold: 0 Whitney Houston correct: False seconds: 0.2
decision: None fallback: False
Benchmark summary: {'model': 'Gemma 4 E2B (4-bit)', 'accuracy': 0.6666666666666666, 'avg_seconds': 0.2, 'fallbacks': 0}
Earned amount: 0 Log path: C:\Users\sjuan\Desktop\NLP\project\nlp-polimillionaire\results\runs\gemma_4_e2b_4-bit_20260525_125643.jsonl

Model: Gemma + Qwen Mixed Council (4-bit)


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


...

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.



Q: What was Whitney Houston's debut album?
predicted: 1 Just Whitney
gold: 0 Whitney Houston correct: False seconds: 5.05
decision: judge fallback: False
vote 1 google/gemma-4-E2B-it -> 2 confidence: 0.95 reason: Whitney
vote 2 Qwen/Qwen3.5-2B -> 1 confidence: 0.95 reason: Whitney Houston
judge raw: 1 scope: candidate_only
Benchmark summary: {'model': 'Gemma + Qwen Mixed Council (4-bit)', 'accuracy': 0.6666666666666666, 'avg_seconds': 5.01, 'fallbacks': 0}


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Earned amount: 1000 Log path: C:\Users\sjuan\Desktop\NLP\project\nlp-polimillionaire\results\runs\gemma_qwen_mixed_council_4-bit_20260525_125729.jsonl

Comparison:
{'model': 'Gemma 4 E2B (4-bit)', 'accuracy': 0.6666666666666666, 'avg_seconds': 0.2, 'fallbacks': 0}
{'model': 'Gemma + Qwen Mixed Council (4-bit)', 'accuracy': 0.6666666666666666, 'avg_seconds': 5.01, 'fallbacks': 0}


[{'label': 'Gemma 4 E2B (4-bit)',
  'warmup_option': 1,
  'benchmark_ok': True,
  'earned_amount': 0,
  'log_path': 'C:\\Users\\sjuan\\Desktop\\NLP\\project\\nlp-polimillionaire\\results\\runs\\gemma_4_e2b_4-bit_20260525_125643.jsonl'},
 {'label': 'Gemma + Qwen Mixed Council (4-bit)',
  'warmup_option': 1,
  'benchmark_ok': True,
  'earned_amount': 1000,
  'log_path': 'C:\\Users\\sjuan\\Desktop\\NLP\\project\\nlp-polimillionaire\\results\\runs\\gemma_qwen_mixed_council_4-bit_20260525_125729.jsonl'}]

## Results

In [6]:
from polimillionaire.runner import load_jsonl, summarize_attempts

run_logs = sorted((REPO_ROOT / "results" / "runs").glob("*.jsonl"), key=lambda path: path.stat().st_mtime)
for path in run_logs[-5:]:
    print(path.name)
if run_logs:
    print("latest:", run_logs[-1].name)
    print(summarize_attempts(load_jsonl(run_logs[-1])))

gemma_4_e2b_council_20260525_112330.jsonl
gemma_qwen_mixed_council_20260525_113632.jsonl
gemma_qwen_mixed_council_20260525_115053.jsonl
gemma_4_e2b_4-bit_20260525_125643.jsonl
gemma_qwen_mixed_council_4-bit_20260525_125729.jsonl
latest: gemma_qwen_mixed_council_4-bit_20260525_125729.jsonl
{'total': 6, 'correct': 5, 'accuracy': 0.8333333333333334, 'timed_out': 0, 'avg_elapsed_seconds': 6.210999999995693}
